In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb

file_path = '/content/kat_sale_citrus_merged.csv'

df = pd.read_csv(file_path, encoding='utf-8')

/tmp/ipykernel_675/709385349.py:9: DtypeWarning: Columns (9,13,15,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8')


In [ ]:
# 데이터프레임이 잘 로드되었는지 상위 5개 행을 확인합니다.
df.head()

,avgprc,corp_cd,corp_nm,gds_lclsf_cd,gds_lclsf_nm,gds_mclsf_cd,gds_mclsf_nm,gds_sclsf_cd,gds_sclsf_nm,grd_cd,...,unit_nm,unit_qty,unit_tot_qty,whsl_mrkt_cd,whsl_mrkt_nm,query_whsl_mrkt_cd,query_whsl_mrkt_nm,query_trd_date,query_start_date,query_end_date
0,6519.048,21000901,동부청과㈜,6,과실류,14,감귤,1,조생귤,12,...,kg,5.0,4485.0,210009,부산반여,210009,부산반여,2021-01-04,2021-01-01,2021-01-31
1,7994.030,21000901,동부청과㈜,6,과실류,14,감귤,1,조생귤,12,...,kg,5.0,12630.0,210009,부산반여,210009,부산반여,2021-01-04,2021-01-01,2021-01-31
2,15383.333,21000901,동부청과㈜,6,과실류,14,감귤,1,조생귤,12,...,kg,10.0,4140.0,210009,부산반여,210009,부산반여,2021-01-04,2021-01-01,2021-01-31
3,17733.333,21000901,동부청과㈜,6,과실류,14,감귤,7,NaN,12,...,kg,3.0,480.0,210009,부산반여,210009,부산반여,2021-01-04,2021-01-01,2021-01-31
4,18818.182,21000901,동부청과㈜,6,과실류,14,감귤,7,NaN,12,...,kg,3.0,1494.0,210009,부산반여,210009,부산반여,2021-01-04,2021-01-01,2021-01-31


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728626 entries, 0 to 1728625
Data columns (total 33 columns):
 #   Column              Dtype  
---  ------              -----  
 0   avgprc              float64
 1   corp_cd             int64  
 2   corp_nm             object 
 3   gds_lclsf_cd        int64  
 4   gds_lclsf_nm        object 
 5   gds_mclsf_cd        int64  
 6   gds_mclsf_nm        object 
 7   gds_sclsf_cd        int64  
 8   gds_sclsf_nm        object 
 9   grd_cd              object 
 10  grd_nm              object 
 11  hgprc               float64
 12  lwprc               float64
 13  pkg_cd              object 
 14  pkg_nm              object 
 15  plor_cd             object 
 16  plor_nm             object 
 17  sz_cd               object 
 18  sz_nm               object 
 19  totprc              float64
 20  trd_clcln_ymd       object 
 21  trd_se              object 
 22  unit_cd             int64  
 23  unit_nm             object 
 24  unit_qty            floa

In [ ]:
cols_to_drop = [
    'corp_cd', 'corp_nm',
    'gds_lclsf_cd', 'gds_lclsf_nm',
    'gds_mclsf_cd',
    'grd_cd', 'pkg_cd', 'plor_cd', 'plor_nm', 'sz_cd', 'unit_cd',
    'unit_nm', 'unit_tot_qty', 'whsl_mrkt_cd', 'whsl_mrkt_nm',
    'query_whsl_mrkt_cd', 'query_trd_date', 'query_start_date', 'query_end_date'
]

# 실제 지정한 열들을 데이터프레임에서 제거하여 반영합니다.
existing_cols = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols)

print(f"성공적으로 삭제된 열: {existing_cols}")
print(f"남은 열 개수: {df.shape[1]}")
df.head()

성공적으로 삭제된 열: ['corp_cd', 'corp_nm', 'gds_lclsf_cd', 'gds_lclsf_nm', 'gds_mclsf_cd', 'grd_cd', 'pkg_cd', 'plor_cd', 'plor_nm', 'sz_cd', 'unit_cd', 'unit_nm', 'unit_tot_qty', 'whsl_mrkt_cd', 'whsl_mrkt_nm', 'query_whsl_mrkt_cd', 'query_trd_date', 'query_start_date', 'query_end_date']
남은 열 개수: 14


,avgprc,gds_mclsf_nm,gds_sclsf_cd,gds_sclsf_nm,grd_nm,hgprc,lwprc,pkg_nm,sz_nm,totprc,trd_clcln_ymd,trd_se,unit_qty,query_whsl_mrkt_nm
0,6519.048,감귤,1,조생귤,상,10700.0,4000.0,상자,.,6154700.0,2021-01-04,경매,5.0,부산반여
1,7994.030,감귤,1,조생귤,상,13400.0,4000.0,상자,.,21994500.0,2021-01-04,경매,5.0,부산반여
2,15383.333,감귤,1,조생귤,상,28000.0,6200.0,상자,.,5875500.0,2021-01-04,경매,10.0,부산반여
3,17733.333,감귤,7,NaN,상,19500.0,15200.0,상자,.,2871800.0,2021-01-04,경매,3.0,부산반여
4,18818.182,감귤,7,NaN,상,21700.0,16000.0,상자,.,8903900.0,2021-01-04,경매,3.0,부산반여


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728626 entries, 0 to 1728625
Data columns (total 14 columns):
 #   Column              Dtype  
---  ------              -----  
 0   avgprc              float64
 1   gds_mclsf_nm        object 
 2   gds_sclsf_cd        int64  
 3   gds_sclsf_nm        object 
 4   grd_nm              object 
 5   hgprc               float64
 6   lwprc               float64
 7   pkg_nm              object 
 8   sz_nm               object 
 9   totprc              float64
 10  trd_clcln_ymd       object 
 11  trd_se              object 
 12  unit_qty            float64
 13  query_whsl_mrkt_nm  object 
dtypes: float64(5), int64(1), object(8)
memory usage: 184.6+ MB


In [ ]:
import pandas as pd

# gds_sclsf_cd(소분류 코드)와 gds_sclsf_nm(소분류명) 두 칼럼의 매칭 관계 확인
print("=== 품종 소분류 코드 및 품종명 매칭 분포 ===")
# 결측치(NaN)도 포함하여 함께 그룹바이하여 카운트합니다.
matching_df = df.groupby(['gds_sclsf_cd', 'gds_sclsf_nm'], dropna=False).size().reset_index(name='데이터 수 (건)')

# 보기 편하게 데이터 수 기준 내림차순 정렬
matching_df = matching_df.sort_values(by='데이터 수 (건)', ascending=False)
display(matching_df)

=== 품종 소분류 코드 및 품종명 매칭 분포 ===


,gds_sclsf_cd,gds_sclsf_nm,데이터 수 (건)
10,5,하우스감귤,323034
2,1,조생귤,317692
34,99,기타,306200
8,4,한라봉,196126
14,7,NaN,133411
11,6,극조생감귤,92637
7,4,비가림감귤,63332
13,7,천혜향,59701
18,9,NaN,53092
16,8,NaN,48678


In [ ]:
# 딕셔너리를 정의하여 특정 코드에 대응하는 이름을 지정합니다.
mapping_dict = {
    7: '천혜향',
    8: '황금향',
    9: '레드향',
    10: '한라향',
    11: '카라향'
}

# 결측치(NaN)인 행들에 대해서만 gds_sclsf_cd 값에 따라 gds_sclsf_nm을 채워줍니다.
is_na = df['gds_sclsf_nm'].isna()
df.loc[is_na, 'gds_sclsf_nm'] = df.loc[is_na, 'gds_sclsf_cd'].map(mapping_dict)

# 반영 후 결측치 수 및 매칭 분포 재확인
print("=== 결측치 대체 후 남은 결측치 수 ===")
print(df['gds_sclsf_nm'].isna().sum())

print("\n=== 업데이트된 품종 소분류 코드 및 품종명 매칭 분포 ===")
updated_matching_df = df.groupby(['gds_sclsf_cd', 'gds_sclsf_nm'], dropna=False).size().reset_index(name='데이터 수 (건)')
display(updated_matching_df.sort_values(by='데이터 수 (건)', ascending=False))

=== 결측치 대체 후 남은 결측치 수 ===
0

=== 업데이트된 품종 소분류 코드 및 품종명 매칭 분포 ===


,gds_sclsf_cd,gds_sclsf_nm,데이터 수 (건)
10,5,하우스감귤,323034
2,1,조생귤,317692
29,99,기타,306200
8,4,한라봉,196126
13,7,천혜향,193112
11,6,극조생감귤,92637
15,9,레드향,90592
7,4,비가림감귤,63332
14,8,황금향,58238
22,19,타이벡감귤,19719


In [ ]:
df.head()

,avgprc,gds_mclsf_nm,gds_sclsf_cd,gds_sclsf_nm,grd_nm,hgprc,lwprc,pkg_nm,sz_nm,totprc,trd_clcln_ymd,trd_se,unit_qty,query_whsl_mrkt_nm
0,6519.048,감귤,1,조생귤,상,10700.0,4000.0,상자,.,6154700.0,2021-01-04,경매,5.0,부산반여
1,7994.030,감귤,1,조생귤,상,13400.0,4000.0,상자,.,21994500.0,2021-01-04,경매,5.0,부산반여
2,15383.333,감귤,1,조생귤,상,28000.0,6200.0,상자,.,5875500.0,2021-01-04,경매,10.0,부산반여
3,17733.333,감귤,7,천혜향,상,19500.0,15200.0,상자,.,2871800.0,2021-01-04,경매,3.0,부산반여
4,18818.182,감귤,7,천혜향,상,21700.0,16000.0,상자,.,8903900.0,2021-01-04,경매,3.0,부산반여


In [ ]:
# 1. unit_qty(수량/중량) 구간 정의
qty_bins = [0, 3.5, 7.5, 12, 1000]  # 구간 경계값
qty_labels = ['소형(3.5kg이하)', '중형(3.5kg초과~7.5kg이하)', '대형(7.5kg초과~12kg이하)', '가공용(12kg초과)'] # 각 구간의 범주 이름

# 2. gds_sclsf_cd(소분류 코드) 매핑
custom_code_map = {
    1: '노지온주',
    6: '노지온주',
    0: '노지온주',
    2: '노지온주',
    3: '노지온주',
    13: '노지온주',
    32: '노지온주',
    19: '노지온주',
    5: '하우스감귤',
    4: '비가림감귤',
    7: '만감류',
    8: '만감류',
    9: '만감류',
    10: '만감류',
    11: '만감류',
    12: '만감류',
    15: '만감류',
    17: '만감류',
    34: '만감류',
    23: '만감류',
    98: '수입',
    20: '8~9월 풋귤(청귤)',
    99: '기타'
}

# =========================================================
# 설정한 기준으로 범주형 컬럼 생성 및 변환 적용
# =========================================================

# 1. unit_qty 범주화 적용
df['unit_qty_grp'] = pd.cut(df['unit_qty'], bins=qty_bins, labels=qty_labels, include_lowest=True)
df['unit_qty_grp'] = df['unit_qty_grp'].astype('category')

# 2. gds_mclsf_nm이 '만감'인 경우 '만감류'로 먼저 할당하고, 나머지는 custom_code_map에 맞춰 매핑합니다.
df['gds_sclsf_grp'] = np.where(
    df['gds_mclsf_nm'] == '만감',
    '만감류',
    df['gds_sclsf_cd'].map(custom_code_map).fillna('감귤')
)
df['gds_sclsf_grp'] = df['gds_sclsf_grp'].astype('category')

# 적용 결과 확인
print("=== [1] unit_qty 사용자 지정 범주 분포 ===")
print(df['unit_qty_grp'].value_counts(dropna=False))

print("\n=== [2] gds_sclsf_cd/gds_mclsf_nm 기반 사용자 지정 범주 분포 ===")
print(df['gds_sclsf_grp'].value_counts(dropna=False))


# 데이터프레임 확인
display(df[['gds_mclsf_nm', 'gds_sclsf_cd', 'gds_sclsf_grp']].drop_duplicates().head(15))


=== [1] unit_qty 사용자 지정 범주 분포 ===
unit_qty_grp
중형(3.5kg초과~7.5kg이하)    863959
소형(3.5kg이하)            598196
대형(7.5kg초과~12kg이하)     264391
가공용(12kg초과)              2080
Name: count, dtype: int64

=== [2] gds_sclsf_cd/gds_mclsf_nm 기반 사용자 지정 범주 분포 ===
gds_sclsf_grp
만감류            728666
노지온주           450506
하우스감귤          323034
기타             161405
비가림감귤           63332
수입               1659
8~9월 풋귤(청귤)        24
Name: count, dtype: int64


,gds_mclsf_nm,gds_sclsf_cd,gds_sclsf_grp
0,감귤,1,노지온주
3,감귤,7,만감류
16,감귤,99,기타
69,감귤,4,비가림감귤
80,감귤,5,하우스감귤
797,감귤,9,만감류
830,감귤,0,노지온주
10104,감귤,15,만감류
11699,감귤,3,노지온주
13412,감귤,98,수입


In [ ]:
# unit_qty(거래수량/중량)의 고유값 종류와 빈도수 확인
print("=== unit_qty 고유값 수 및 기술통계 ===")
print(f"고유값 종류 개수: {df['unit_qty'].nunique()}개")

qty_counts = df['unit_qty'].value_counts().reset_index()
qty_counts.columns = ['unit_qty (중량)', '데이터 수 (건)']
display(qty_counts.head(139))

=== unit_qty 고유값 수 및 기술통계 ===
고유값 종류 개수: 169개


,unit_qty (중량),데이터 수 (건)
0,5.00,827395
1,3.00,537022
2,9.00,139539
3,10.00,115953
4,0.50,41182
...,...,...
134,0.55,2
135,4.87,2
136,3.90,2
137,5.04,2


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728626 entries, 0 to 1728625
Data columns (total 16 columns):
 #   Column              Dtype   
---  ------              -----   
 0   avgprc              float64 
 1   gds_mclsf_nm        object  
 2   gds_sclsf_cd        int64   
 3   gds_sclsf_nm        object  
 4   grd_nm              object  
 5   hgprc               float64 
 6   lwprc               float64 
 7   pkg_nm              object  
 8   sz_nm               object  
 9   totprc              float64 
 10  trd_clcln_ymd       object  
 11  trd_se              object  
 12  unit_qty            float64 
 13  query_whsl_mrkt_nm  object  
 14  unit_qty_grp        category
 15  gds_sclsf_grp       category
dtypes: category(2), float64(5), int64(1), object(8)
memory usage: 187.9+ MB


In [ ]:
# grd_nm(등급명) 컬럼의 고유값 종류 및 빈도수 확인
print("=== 등급명(grd_nm) 분포 확인 (결측치 포함) ===")
grade_counts = df['grd_nm'].value_counts(dropna=False).reset_index()
grade_counts.columns = ['등급명 (grd_nm)', '데이터 수 (건)']
display(grade_counts)

=== 등급명(grd_nm) 분포 확인 (결측치 포함) ===


,등급명 (grd_nm),데이터 수 (건)
0,특,1312235
1,.,126850
2,중,91151
3,상,77216
4,등외,48647
5,하,32430
6,5등,17131
7,7등,8540
8,6등,8137
9,8등,4384


In [ ]:
# sz_nm(크기) 컬럼의 고유값 종류 및 빈도수 확인
print("=== 크기(sz_nm) 분포 확인 (결측치 포함) ===")
grade_counts = df['sz_nm'].value_counts(dropna=False).reset_index()
grade_counts.columns = ['크기 (sz_nm)', '데이터 수 (건)']
display(grade_counts)

=== 크기(sz_nm) 분포 확인 (결측치 포함) ===


,크기 (sz_nm),데이터 수 (건)
0,.,668239
1,50내,80073
2,40내,78895
3,12개,53079
4,10개,51930
...,...,...
64,200내,251
65,190내,147
66,72개,93
67,113개,47


In [ ]:
# trd_se(거래방법) 컬럼의 고유값 종류 및 빈도수 확인
grade_counts = df['trd_se'].value_counts(dropna=False).reset_index()
grade_counts.columns = ['거래방법 (trd_se)', '데이터 수 (건)']
display(grade_counts)

,거래방법 (trd_se),데이터 수 (건)
0,경매,1467506
1,정가수의,226261
2,정가수의(예약형),17682
3,전자거래(팩스형),7106
4,전자거래,3856
5,-,3028
6,자기계산,2934
7,온라인경매(입찰식),252
8,온라인경매(응찰식),1


In [ ]:
from google.colab import files

# 1. '정가수의' 및 '정가수의(예약형)' 데이터만 필터링
target_trd_se = ['경매']
filtered_df = df[df['trd_se'].isin(target_trd_se)]

# # 2. 필터링 결과 요약 출력
# print(f"필터링 전 전체 행 수: {df.shape[0]:,}개")
# print(f"필터링 후 (정가수의 + 예약형) 행 수: {filtered_df.shape[0]:,}개")

# # # 3. CSV 파일로 로컬에 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 적용)
# # output_file_path = '/content/kat_sale_citrus_fixed_price.csv'
# # filtered_df.to_csv(output_file_path, index=False, encoding='utf-8-sig')
# # print(f"파일이 {output_file_path} 경로에 저장되었습니다.")

# # # 4. 코랩 브라우저를 통해 파일 자동 다운로드
# # files.download(output_file_path)

필터링 전 전체 행 수: 1,728,626개
필터링 후 (정가수의 + 예약형) 행 수: 1,467,506개


In [ ]:
cols_to_drop = ['gds_mclsf_nm']
existing_cols = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols)